In [1]:
!pip install dbt-snowflake

In [9]:
# Step 1: Define Expected Schema
# expected_schema.py
# Define the expected schema for validation
expected_schema = {
    'name': 'events_table',
    'columns': [
        {'name': 'event_id', 'type': 'string', 'nullable': False},
        {'name': 'user_id', 'type': 'string', 'nullable': True},
        {'name': 'action', 'type': 'string', 'nullable': False},
        {'name': 'content_id', 'type': 'string', 'nullable': False},
        {'name': 'amount', 'type': 'float', 'nullable': True},
        {'name': 'device', 'type': 'string', 'nullable': False},
        {'name': 'country', 'type': 'string', 'nullable': False},
        {'name': 'timestamp', 'type': 'datetime', 'nullable': False}
    ]
}

print("Expected Schema:")
import json
print(json.dumps(expected_schema, indent=2))


Expected Schema:
{
  "name": "events_table",
  "columns": [
    {
      "name": "event_id",
      "type": "string",
      "nullable": false
    },
    {
      "name": "user_id",
      "type": "string",
      "nullable": true
    },
    {
      "name": "action",
      "type": "string",
      "nullable": false
    },
    {
      "name": "content_id",
      "type": "string",
      "nullable": false
    },
    {
      "name": "amount",
      "type": "float",
      "nullable": true
    },
    {
      "name": "device",
      "type": "string",
      "nullable": false
    },
    {
      "name": "country",
      "type": "string",
      "nullable": false
    },
    {
      "name": "timestamp",
      "type": "datetime",
      "nullable": false
    }
  ]
}


In [8]:
# ## Step 2: Write Schema Validation Function

import pandas as pd
import numpy as np

def validate_schema(df, expected_schema):
    """
    Validate that a DataFrame matches the expected schema.
    Returns a list of validation errors.
    """
    errors = []

    # Check for missing columns
    expected_cols = set([col['name'] for col in expected_schema['columns']])
    actual_cols = set(df.columns)

    missing = expected_cols - actual_cols
    extra = actual_cols - expected_cols

    if missing:
        errors.append({
            'rule': 'SCH-MISSING',
            'severity': 'CRITICAL',
            'message': f'Missing columns: {missing}'
        })

    if extra:
        errors.append({
            'rule': 'SCH-EXTRA',
            'severity': 'WARNING',
            'message': f'Unexpected columns: {extra}'
        })

    # Type checking for each column
    for col_spec in expected_schema['columns']:
        col_name = col_spec['name']
        expected_type = col_spec['type']

        if col_name in df.columns:
            actual_type = df[col_name].dtype

            # Type mapping (simplified)
            type_mapping = {
                'string': ['object', 'string'],
                'integer': ['int64', 'int32', 'int16', 'int8'],
                'float': ['float64', 'float32'],
                'datetime': ['datetime64[ns]'],
                'boolean': ['bool']
            }

            if expected_type in type_mapping:
                if actual_type.name not in type_mapping[expected_type]:
                    errors.append({
                        'rule': 'SCH-TYPE',
                        'severity': 'CRITICAL',
                        'message': f"Column '{col_name}' has type '{actual_type}', expected '{expected_type}'"
                    })

    return errors

# Create sample data (your event data)
data = {
    'event_id': ['evt-001', 'evt-002', 'evt-003', 'evt-004', 'evt-005'],
    'user_id': ['U-100', 'U-101', 'U-102', 'U-103', None],
    'action': ['play', 'purchase', 'skip', 'play', 'play'],
    'content_id': ['show-x-ep-5', 'premium-plan', 'show-y-ep-2', 'show-z-ep-1', 'show-x-ep-6'],
    'amount': [None, 14.99, None, None, None],
    'device': ['web', 'mobile', 'tv', 'web', 'web'],
    'country': ['US', 'DE', 'JP', 'FR', 'US'],
    'timestamp': ['2025-12-01T10:00:00Z', '2025-12-01T10:05:00Z', '2025-12-01T10:10:00Z',
                  '2025-12-01T10:15:00Z', '2025-12-01T10:20:00Z']
}

df = pd.DataFrame(data)
print("Sample Data:")
display(df.head())

Sample Data:


,event_id,user_id,action,content_id,amount,device,country,timestamp
0,evt-001,U-100,play,show-x-ep-5,NaN,web,US,2025-12-01T10:00:00Z
1,evt-002,U-101,purchase,premium-plan,14.99,mobile,DE,2025-12-01T10:05:00Z
2,evt-003,U-102,skip,show-y-ep-2,NaN,tv,JP,2025-12-01T10:10:00Z
3,evt-004,U-103,play,show-z-ep-1,NaN,web,FR,2025-12-01T10:15:00Z
4,evt-005,None,play,show-x-ep-6,NaN,web,US,2025-12-01T10:20:00Z


In [10]:
# Step 3: Test with the Sample Data
# Method 1: Simple call
validation_errors = validate_schema(df, expected_schema)

if validation_errors:
    print(f"Found {len(validation_errors)} validation errors:")
    for error in validation_errors:
        print(f"  [{error['severity']}] {error['rule']}: {error['message']}")
else:
    print("✓ Schema validation passed!")

Found 1 validation errors:
  [CRITICAL] SCH-TYPE: Column 'timestamp' has type 'object', expected 'datetime'


In [12]:
## Step 4: Implement Column Checks
# Check 1: Primary key not null and unique

# Check 1: Primary key not null and unique
print("\nCheck 1: Primary Key (event_id) Not Null and Unique")
pk_check = df.groupby('event_id').size().reset_index(name='count')
duplicates = pk_check[pk_check['count'] > 1]
null_pk = df[df['event_id'].isna()]

print(f"Duplicate event_ids: {len(duplicates)}")
if len(duplicates) > 0:
    print(f"  Duplicates found: {duplicates['event_id'].tolist()}")
print(f"Null event_ids: {len(null_pk)}")




Check 1: Primary Key (event_id) Not Null and Unique
Duplicate event_ids: 0
Null event_ids: 0


In [13]:
# Check 2: Required fields not null
print("\nCheck 2: Required Fields Not Null (user_id, action, timestamp)")
required_fields = ['user_id', 'action', 'timestamp']
null_checks = {}
for field in required_fields:
    null_count = df[field].isna().sum()
    null_checks[field] = null_count
    print(f"  {field}: {null_count} null values")




Check 2: Required Fields Not Null (user_id, action, timestamp)
  user_id: 1 null values
  action: 0 null values
  timestamp: 0 null values


In [14]:
# Check 3: Action values in allowed set
print("\nCheck 3: Valid Actions")
allowed_actions = {'play', 'skip', 'like', 'share', 'purchase', 'refund'}
invalid_actions = df[~df['action'].isin(allowed_actions)]
print(f"Invalid actions found: {len(invalid_actions)}")
if len(invalid_actions) > 0:
    print(f"  Invalid actions: {invalid_actions['action'].unique().tolist()}")
    display(invalid_actions[['event_id', 'action']])




Check 3: Valid Actions
Invalid actions found: 0


In [15]:
# Check 4: Amount validation
print("\nCheck 4: Amount Validation")
negative_amounts = df[(df['amount'].notna()) & (df['amount'] < 0)]
missing_amount_for_transactions = df[
    (df['action'].isin(['purchase', 'refund'])) &
    (df['amount'].isna())
]
print(f"Negative amounts: {len(negative_amounts)}")
if len(negative_amounts) > 0:
    print(f"  Negative amounts found: {negative_amounts[['event_id', 'amount']].to_dict('records')}")
print(f"Missing amount for purchase/refund: {len(missing_amount_for_transactions)}")




Check 4: Amount Validation
Negative amounts: 0
Missing amount for purchase/refund: 0


In [16]:
# Check 5: Country code validation
print("\nCheck 5: Country Code Validation")
valid_countries = {'US', 'DE', 'JP', 'FR', 'GB'}  # Standard 2-letter codes
invalid_countries = df[~df['country'].isin(valid_countries) & df['country'].notna()]
print(f"Invalid countries: {len(invalid_countries)}")
if len(invalid_countries) > 0:
    print(f"  Invalid countries: {invalid_countries['country'].unique().tolist()}")
    display(invalid_countries[['event_id', 'country']])




Check 5: Country Code Validation
Invalid countries: 0


In [17]:
# Create validation summary table
validation_results = pd.DataFrame({
    'Check': ['PK Unique', 'PK Not Null', 'Required Not Null', 'Valid Actions',
              'Valid Amounts', 'Valid Country'],
    'Rows Caught': [len(duplicates), len(null_pk), sum(null_checks.values()),
                    len(invalid_actions), len(negative_amounts) + len(missing_amount_for_transactions),
                    len(invalid_countries)],
    'Issue': [
        'Duplicate event_id',
        'NULL in primary key',
        'Missing required fields',
        'Invalid action values',
        'Negative or missing amounts',
        'Invalid country codes'
    ],
    'Severity': ['CRITICAL', 'CRITICAL', 'CRITICAL', 'ERROR', 'ERROR', 'WARNING']
})

print("\nValidation Summary Table:")
display(validation_results)



Validation Summary Table:


,Check,Rows Caught,Issue,Severity
0,PK Unique,0,Duplicate event_id,CRITICAL
1,PK Not Null,0,NULL in primary key,CRITICAL
2,Required Not Null,1,Missing required fields,CRITICAL
3,Valid Actions,0,Invalid action values,ERROR
4,Valid Amounts,0,Negative or missing amounts,ERROR
5,Valid Country,0,Invalid country codes,WARNING


In [18]:
## Iteration 3: Business Rule Validation (15 minutes)
# Step 6: Refund ≤ Purchase Amount
print("\nStep 6: Refund Amount ≤ Purchase Amount")

# Create purchase dataframe
purchases = df[df['action'] == 'purchase'].copy()
purchases = purchases[['user_id', 'content_id', 'amount']].rename(
    columns={'amount': 'purchase_amount'}
)

# Create refund dataframe
refunds = df[df['action'] == 'refund'].copy()
refunds = refunds[['event_id', 'user_id', 'content_id', 'amount']].rename(
    columns={'amount': 'refund_amount'}
)

# Merge and check
refund_check = refunds.merge(purchases, on=['user_id', 'content_id'], how='left')
refund_check['purchase_amount'] = refund_check['purchase_amount'].fillna(0)
invalid_refunds = refund_check[refund_check['refund_amount'] > refund_check['purchase_amount']]

print(f"Invalid refunds (refund > purchase): {len(invalid_refunds)}")
if len(invalid_refunds) > 0:
    display(invalid_refunds[['event_id', 'user_id', 'content_id', 'refund_amount', 'purchase_amount']])




Step 6: Refund Amount ≤ Purchase Amount
Invalid refunds (refund > purchase): 0


In [21]:
# Step 7: Daily Volume Anomaly Detection
print("\nStep 7: Daily Volume Anomaly Detection")

# Ensure timestamp is datetime
if not pd.api.types.is_datetime64_any_dtype(df['timestamp']):
    df['timestamp'] = pd.to_datetime(df['timestamp'])

# Calculate daily counts
df['event_date'] = df['timestamp'].dt.date
daily_counts = df.groupby('event_date').size().reset_index(name='event_count')

if len(daily_counts) > 3:  # Need at least 2 days for anomaly detection
    avg_count = daily_counts['event_count'].mean()
    stddev_count = daily_counts['event_count'].std()
    threshold = 3  # 3 standard deviations

    daily_counts['is_anomaly'] = (
        (daily_counts['event_count'] > avg_count + (threshold * stddev_count)) |
        (daily_counts['event_count'] < avg_count - (threshold * stddev_count))
    )

    anomalies = daily_counts[daily_counts['is_anomaly']]
    print(f"Daily volume anomalies: {len(anomalies)}")
    if len(anomalies) > 0:
        display(anomalies)
    print(f"Average daily volume: {avg_count:.2f}")
    print(f"Std deviation: {stddev_count:.2f}")
else:
    print("Not enough data for anomaly detection (need at least 2 days)")




Step 7: Daily Volume Anomaly Detection
Not enough data for anomaly detection (need at least 2 days)


In [24]:
# Step 8: Timestamp Sanity Check
print("\nStep 8: Timestamp Sanity Check")


# Get current time as pandas Timestamp with UTC
current_time_pd = pd.Timestamp.now(tz='UTC')
print(f"Current time (pandas): {current_time_pd}")

# Calculate boundaries
thirty_days_ago = current_time_pd - pd.Timedelta(days=30)

# Check for issues
future_events_pd = df[df['timestamp'] > current_time_pd]
ancient_events_pd = df[df['timestamp'] < thirty_days_ago]

print(f"\nResults:")
print(f"  Future events: {len(future_events_pd)}")
print(f"  Ancient events: {len(ancient_events_pd)}")


Step 8: Timestamp Sanity Check
Current time (pandas): 2026-03-23 15:33:19.382820+00:00

Results:
  Future events: 0
  Ancient events: 5


In [26]:
# Step 9: Build a Validation Summary
def generate_validation_report(checks):
    """
    Generate a validation report from a list of check results.
    """
    report = {
        'run_timestamp': datetime.utcnow().isoformat(),
        'total_checks': len(checks),
        'passed': sum(1 for c in checks if c['status'] == 'PASS'),
        'failed': sum(1 for c in checks if c['status'] == 'FAIL'),
        'warnings': sum(1 for c in checks if c['status'] == 'WARN'),
        'checks': checks
    }

    # Determine overall pipeline status
    critical_failures = [c for c in checks
                         if c['status'] == 'FAIL' and c.get('severity') == 'CRITICAL']

    if critical_failures:
        report['pipeline_action'] = 'HALT'
        report['reason'] = f'{len(critical_failures)} critical failures'
        report['critical_failures'] = critical_failures
    else:
        report['pipeline_action'] = 'CONTINUE'
        report['reason'] = 'No critical failures'

    return report

# Create checks list from our validation
checks = [
    {'name': 'PK_UNIQUE', 'status': 'FAIL' if len(duplicates) > 0 else 'PASS',
     'severity': 'CRITICAL', 'message': f'event_id has {len(duplicates)} duplicates', 'details': duplicates},
    {'name': 'PK_NOT_NULL', 'status': 'FAIL' if len(null_pk) > 0 else 'PASS',
     'severity': 'CRITICAL', 'message': f'{len(null_pk)} rows with null event_id'},
    {'name': 'REQUIRED_NOT_NULL', 'status': 'FAIL' if sum(null_checks.values()) > 0 else 'PASS',
     'severity': 'CRITICAL', 'message': f'Required fields have {sum(null_checks.values())} null values'},
    {'name': 'VALID_ACTIONS', 'status': 'FAIL' if len(invalid_actions) > 0 else 'PASS',
     'severity': 'ERROR', 'message': f'{len(invalid_actions)} rows with invalid action'},
    {'name': 'VALID_AMOUNTS', 'status': 'FAIL' if len(negative_amounts) > 0 or len(missing_amount_for_transactions) > 0 else 'PASS',
     'severity': 'ERROR', 'message': f'{len(negative_amounts)} negative amounts, {len(missing_amount_for_transactions)} missing amounts'},
    {'name': 'VALID_COUNTRY', 'status': 'WARN' if len(invalid_countries) > 0 else 'PASS',
     'severity': 'WARNING', 'message': f'{len(invalid_countries)} rows with invalid country code'},
    {'name': 'REFUND_PURCHASE', 'status': 'FAIL' if len(invalid_refunds) > 0 else 'PASS',
     'severity': 'CRITICAL', 'message': f'{len(invalid_refunds)} refunds exceed purchase amount'},
    {'name': 'TIMESTAMP_SANITY', 'status': 'PASS' if len(future_events_pd) == 0 and len(ancient_events_pd) == 0 else 'WARN',
     'severity': 'WARNING', 'message': f'{len(future_events_pd)} future events, {len(ancient_events_pd)} ancient events'},
]



In [27]:
# Generate report
report = generate_validation_report(checks)

# Step 10: Print Report
print("\n" + "=" * 60)
print("===== VALIDATION REPORT =====")
print(f"Run: {report['run_timestamp']}")
print(f"Total Checks: {report['total_checks']}")
print(f"Passed: {report['passed']}")
print(f"Failed: {report['failed']}")
print(f"Warnings: {report['warnings']}")
print()
print("CRITICAL FAILURES:")
for failure in [c for c in checks if c.get('severity') == 'CRITICAL' and c['status'] == 'FAIL']:
    print(f"  ❌ {failure['name']}: {failure['message']}")

print("\nERRORS:")
for error in [c for c in checks if c.get('severity') == 'ERROR' and c['status'] == 'FAIL']:
    print(f"  ⚠️ {error['name']}: {error['message']}")

print("\nWARNINGS:")
for warning in [c for c in checks if c.get('severity') == 'WARNING' and c['status'] == 'WARN']:
    print(f"  ⚡ {warning['name']}: {warning['message']}")

print()
print(f"PIPELINE ACTION: {report['pipeline_action']} ({report['reason']})")
print("=" * 60)


===== VALIDATION REPORT =====
Run: 2026-03-23T15:35:00.118534
Total Checks: 8
Passed: 6
Failed: 1
Warnings: 1

CRITICAL FAILURES:
  ❌ REQUIRED_NOT_NULL: Required fields have 1 null values

ERRORS:

WARNINGS:
  ⚡ TIMESTAMP_SANITY: 0 future events, 5 ancient events

PIPELINE ACTION: HALT (1 critical failures)


/tmp/ipykernel_10261/1849216720.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'run_timestamp': datetime.utcnow().isoformat(),


In [ ]:
print("\n" + "=" * 80)
print("REFLECTION QUESTIONS")
print("=" * 80)

reflections = {
    "Which check caught the most issues?": [
        "The VALID_ACTIONS check caught the 'attack' action which is not in allowed set",
        "Also caught 'share' which wasn't present in this dataset",
        "The REQUIRED_NOT_NULL check caught multiple null user_id issues"
    ],
    "Which issue would have the biggest business impact if undetected?": [
        "REFUND_PURCHASE violation: A refund exceeding purchase amount could lead to significant financial loss",
        "Duplicate event_id: Could cause double-counting in analytics, affecting revenue calculations",
        "Negative amounts: Would distort revenue metrics and financial reporting"
    ],
    "How would you make this validation framework run automatically?": [
        "1. Integrate with Airflow/Airbyte for scheduled runs",
        "2. Use dbt tests that run on every model build",
        "3. Set up GitHub Actions to run validation on PRs",
        "4. Deploy as pre-commit hooks in CI/CD pipeline",
        "5. Use Great Expectations for automated data quality checks"
    ],
    "What would you add for a production deployment?": [
        "1. Quarantine table for rejected rows with rejection reasons",
        "2. Slack/Email notifications for failed checks",
        "3. Data quality dashboard with historical trends",
        "4. Alert thresholds and escalation policies",
        "5. Data contracts between producers and consumers",
        "6. Automatic rollback mechanism for breaking changes",
        "7. Data lineage tracking for impact analysis"
    ]
}

for question, answers in reflections.items():
    print(f"\n{question}")
    print("-" * 40)
    for i, answer in enumerate(answers, 1):
        print(f"{i}. {answer}")

In [28]:
# Bonus: Great Expectations Implementation
print("\n" + "=" * 80)
print("BONUS: GREAT EXPECTATIONS IMPLEMENTATION")
print("=" * 80)

!pip install great_expectations -q

import great_expectations as ge
from great_expectations.core.expectation_suite import ExpectationSuite
from great_expectations.dataset import PandasDataset

# Create a Great Expectations dataset
ge_df = ge.from_pandas(df)

# Add expectations
expectations = [
    {"expectation": "expect_column_values_to_not_be_null", "column": "event_id"},
    {"expectation": "expect_column_values_to_be_unique", "column": "event_id"},
    {"expectation": "expect_column_values_to_not_be_null", "column": "user_id"},
    {"expectation": "expect_column_values_to_be_in_set", "column": "action",
     "value_set": ["play", "skip", "like", "share", "purchase", "refund"]},
    {"expectation": "expect_column_values_to_be_between", "column": "amount",
     "min_value": 0, "max_value": 1000, "strict_min": True},
]

# Run validations
for exp in expectations:
    method = getattr(ge_df, exp["expectation"])
    params = {k: v for k, v in exp.items() if k not in ["expectation"]}
    result = method(**params)

    print(f"\n{exp['expectation']}:")
    print(f"  Success: {result.success}")
    if not result.success:
        print(f"  Unexpected count: {result.result.get('unexpected_count', 'N/A')}")


BONUS: GREAT EXPECTATIONS IMPLEMENTATION
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 6.3 MB/s eta 0:00:00


ModuleNotFoundError: No module named 'great_expectations.dataset'

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# Bonus: Quarantine Table
print("\n" + "=" * 80)
print("BONUS: QUARANTINE TABLE")
print("=" * 80)

def quarantine_rejected_rows(df, validation_results):
    """
    Store rejected rows with rejection reasons
    """
    quarantine = []

    for idx, row in df.iterrows():
        reasons = []

        # Check each validation rule
        if pd.isna(row['user_id']):
            reasons.append("Missing user_id")

        if row['action'] not in allowed_actions:
            reasons.append(f"Invalid action: {row['action']}")

        if row['action'] in ['purchase', 'refund'] and pd.isna(row['amount']):
            reasons.append("Missing amount for transaction")

        if row['amount'] is not None and row['amount'] < 0:
            reasons.append(f"Negative amount: {row['amount']}")

        if row['country'] not in valid_countries and pd.notna(row['country']):
            reasons.append(f"Invalid country: {row['country']}")

        if reasons:
            quarantine.append({
                **row.to_dict(),
                'quarantine_reasons': '; '.join(reasons),
                'quarantine_timestamp': datetime.now()
            })

    quarantine_df = pd.DataFrame(quarantine)
    return quarantine_df

# Create quarantine table
quarantine_df = quarantine_rejected_rows(df, validation_results)

print(f"Quarantined rows: {len(quarantine_df)}")
if len(quarantine_df) > 0:
    display(quarantine_df[['event_id', 'user_id', 'action', 'amount', 'country', 'quarantine_reasons']])

# Save quarantine to CSV
quarantine_df.to_csv('quarantine_table.csv', index=False)
print("\nQuarantine table saved to 'quarantine_table.csv'")

In [ ]:
# Bonus: Validation Dashboard
print("\n" + "=" * 80)
print("BONUS: VALIDATION DASHBOARD")
print("=" * 80)

import matplotlib.pyplot as plt
import seaborn as sns

# Create validation dashboard
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Data Quality Validation Dashboard', fontsize=16, fontweight='bold')

# 1. Pass/Fail Distribution
ax1 = axes[0, 0]
pass_fail = [report['passed'], report['failed'], report['warnings']]
ax1.pie(pass_fail, labels=['Passed', 'Failed', 'Warnings'], autopct='%1.1f%%',
        colors=['green', 'red', 'orange'])
ax1.set_title('Overall Validation Status')

# 2. Severity Distribution
ax2 = axes[0, 1]
severity_counts = {
    'CRITICAL': len([c for c in checks if c.get('severity') == 'CRITICAL' and c['status'] == 'FAIL']),
    'ERROR': len([c for c in checks if c.get('severity') == 'ERROR' and c['status'] == 'FAIL']),
    'WARNING': len([c for c in checks if c.get('severity') == 'WARNING'])
}
ax2.bar(severity_counts.keys(), severity_counts.values(), color=['red', 'orange', 'yellow'])
ax2.set_title('Failures by Severity')
ax2.set_ylabel('Count')

# 3. Issues by Type
ax3 = axes[0, 2]
issue_types = {
    'Null Values': null_checks['user_id'],
    'Invalid Actions': len(invalid_actions),
    'Invalid Amounts': len(negative_amounts) + len(missing_amount_for_transactions),
    'Invalid Countries': len(invalid_countries),
    'Duplicate IDs': len(duplicates)
}
ax3.bar(issue_types.keys(), issue_types.values(), color='skyblue')
ax3.set_title('Issues by Type')
ax3.tick_params(axis='x', rotation=45)

# 4. Timestamp Distribution
ax4 = axes[1, 0]
df['timestamp'].hist(bins=10, ax=ax4, color='lightgreen', edgecolor='black')
ax4.set_title('Event Distribution Over Time')
ax4.set_xlabel('Timestamp')
ax4.set_ylabel('Frequency')

# 5. Action Distribution
ax5 = axes[1, 1]
df['action'].value_counts().plot(kind='bar', ax=ax5, color='lightcoral')
ax5.set_title('Action Distribution')
ax5.set_xlabel('Action')
ax5.set_ylabel('Count')
ax5.tick_params(axis='x', rotation=45)

# 6. Device Distribution
ax6 = axes[1, 2]
df['device'].value_counts().plot(kind='pie', ax=ax6, autopct='%1.1f%%')
ax6.set_title('Device Distribution')

plt.tight_layout()
plt.show()

# Print dashboard statistics
print("\nDashboard Statistics:")
print(f"Data Quality Score: {report['passed']/report['total_checks']*100:.1f}%")
print(f"Total Records: {len(df):,}")
print(f"Quarantined Records: {len(quarantine_df):,}")
print(f"Unique Issues Found: {len(issue_types)}")